# Complex Autoencoder Reconstructions

Load the best native-complex autoencoder checkpoint and compare original coil images with reconstructed outputs.

The notebook plots 4 samples from the test split and 2 samples from the training split using HoloViews.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import holoviews as hv
import numpy as np
import pandas as pd
import torch

hv.extension("bokeh")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.autoencoder.dataset import ComplexCoilImageDataset
from src.autoencoder.model import build_autoencoder

## Settings

Change `CHECKPOINT_PATH` if your training output directory is different.

In [ ]:
from pathlib import Path

REPO_ROOT = Path("/home/mad07/Datasets/Prostate MRI")
CHECKPOINT_PATH = REPO_ROOT / "complex_t2_autoencoder" / "best_model.pt"

   #CHECKPOINT_PATH = REPO_ROOT / "runs" / "complex_t2_autoencoder" / "best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_SAMPLES = 4
TRAIN_SAMPLES = 2

print(f"repo root: {REPO_ROOT}")
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"device: {DEVICE}")

## Load Checkpoint, Model, And Data

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}. Train first with: "
        "python -m src.autoencoder.train --config configs/t2_complex_autoencoder.yaml"
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
config = checkpoint["config"]

model = build_autoencoder(config).to(DEVICE)
load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

fresh_model = build_autoencoder(config).to(DEVICE)
fresh_model.eval()

first_weight_name, first_checkpoint_weight = next(iter(checkpoint["model_state_dict"].items()))
first_fresh_weight = fresh_model.state_dict()[first_weight_name].cpu()
weight_delta = torch.mean(torch.abs(first_checkpoint_weight.cpu() - first_fresh_weight)).item()

train_dataset = ComplexCoilImageDataset(config, "train")
test_dataset = ComplexCoilImageDataset(config, "test")

print(f"checkpoint epoch: {checkpoint.get('epoch')}")
print(f"checkpoint best_val_loss: {checkpoint.get('best_val_loss')}")
print(f"strict load missing keys: {load_result.missing_keys}")
print(f"strict load unexpected keys: {load_result.unexpected_keys}")
print(f"first weight tensor: {first_weight_name} {tuple(first_checkpoint_weight.shape)} {first_checkpoint_weight.dtype}")
print(f"mean abs(checkpoint weight - fresh weight): {weight_delta:.6f}")
print(f"train samples: {len(train_dataset)}")
print(f"test samples: {len(test_dataset)}")
print(f"normalization: {config['data']['normalization']}")

## Training Curves

Train and validation metrics are plotted per epoch from `history.json`. Test metrics are only evaluated after training in the current training script, so they are shown as a final reference point or dashed line instead of a per-epoch curve.

In [ ]:




HISTORY_PATH = CHECKPOINT_PATH.parent / "history.json"
SUMMARY_PATH = CHECKPOINT_PATH.parent / "summary.json"


if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Training history not found: {HISTORY_PATH}")

history_payload = pd.read_json(HISTORY_PATH)
history = pd.DataFrame(history_payload["history"].tolist())
summary = pd.read_json(SUMMARY_PATH, typ="series") if SUMMARY_PATH.exists() else pd.Series(dtype=object)

history["epoch"] = history["epoch"].astype(int)
history
display_columns = [
    "epoch",
    "train_loss",
    "val_loss",
    "train_nmse",
    "val_nmse",
    
]
history[display_columns].tail()


In [ ]:
def metric_curve(frame: pd.DataFrame, metric: str, label: str, color: str):
    if metric not in frame:
        return None
    return hv.Curve(frame, kdims="epoch", vdims=metric, label=label).opts(color=color, line_width=2)


def final_metric_marker(frame: pd.DataFrame, value: float | None, metric_name: str, label: str, color: str):
    if value is None or pd.isna(value) or frame.empty:
        return None
    last_epoch = int(frame["epoch"].max())
    marker_data = pd.DataFrame({"epoch": [last_epoch], metric_name: [float(value)]})
    return hv.Scatter(marker_data, kdims="epoch", vdims=metric_name, label=label).opts(
        color=color,
        marker="diamond",
        size=10,
    )


def final_metric_hline(frame: pd.DataFrame, value: float | None, label: str, color: str):
    if value is None or pd.isna(value) or frame.empty:
        return None
    last_epoch = int(frame["epoch"].max())
    return hv.Curve([(0, float(value)), (last_epoch, float(value))], kdims="epoch", vdims="value", label=label).opts(
        color=color,
        line_dash="dashed",
        line_width=2,
    )


def overlay_non_null(items):
    filtered = [item for item in items if item is not None]
    if not filtered:
        return hv.Overlay([])
    overlay = filtered[0]
    for item in filtered[1:]:
        overlay = overlay * item
    return overlay


test_loss = summary.get("test_loss", None)
test_nmse = summary.get("test_nmse", None)
best_epoch = int(summary.get("best_epoch", history["epoch"].iloc[-1])) if len(history) else 0
best_val_loss = summary.get("best_val_loss", None)

loss_plot = overlay_non_null([
    metric_curve(history, "train_loss", "train loss", "#0072B2"),
    metric_curve(history, "val_loss", "val loss", "#D55E00"),
    metric_curve(history, "test_loss", "test loss per epoch", "#009E73"),
    final_metric_hline(history, test_loss, "final test loss", "#009E73"),
    final_metric_marker(history, test_loss, "train_loss", "final test loss", "#009E73"),
    final_metric_marker(history, best_val_loss, "train_loss", "best val loss", "#CC79A7"),
]).opts(width=850, height=350, xlabel="epoch", ylabel="loss", title="Reconstruction Loss")

nmse_plot = overlay_non_null([
    metric_curve(history, "train_nmse", "train NMSE", "#0072B2"),
    metric_curve(history, "val_nmse", "val NMSE", "#D55E00"),
    metric_curve(history, "test_nmse", "test NMSE per epoch", "#009E73"),
    final_metric_hline(history, test_nmse, "final test NMSE", "#009E73"),
    final_metric_marker(history, test_nmse, "train_nmse", "final test NMSE", "#009E73"),
]).opts(width=850, height=350, xlabel="epoch", ylabel="NMSE", title="Normalized MSE")

gap_plot = overlay_non_null([
    metric_curve(history, "val_minus_train_loss", "val - train loss", "#CC79A7"),
    metric_curve(history, "val_minus_train_nmse", "val - train NMSE", "#56B4E9"),
    hv.HLine(0).opts(color="black", line_dash="dotted", line_width=1),
]).opts(width=850, height=300, xlabel="epoch", ylabel="validation minus train", title="Overfitting Gap")

patience_plot = overlay_non_null([
    metric_curve(history, "stale_epochs_after_epoch", "epochs without val improvement", "#E69F00"),
]).opts(width=850, height=250, xlabel="epoch", ylabel="stale epochs", title="Early-Stopping Patience State")

(loss_plot + nmse_plot + gap_plot + patience_plot).cols(1)

## Reconstruct Selected Samples

In [ ]:
def evenly_spaced_indices(length: int, count: int) -> list[int]:
    if length <= 0:
        return []
    if count >= length:
        return list(range(length))
    return np.linspace(0, length - 1, count, dtype=int).tolist()


@torch.no_grad()
def reconstruct_item(dataset: ComplexCoilImageDataset, split: str, index: int) -> dict[str, object]:
    item = dataset[index]
    image = item["image"].unsqueeze(0).to(device=DEVICE, dtype=torch.complex64)
    reconstruction = model(image).squeeze(0).cpu()
    fresh_reconstruction = fresh_model(image).squeeze(0).cpu()
    normalized_original = item["image"].cpu()
    normalized_error = reconstruction - normalized_original
    normalized_fresh_error = fresh_reconstruction - normalized_original
    normalized_denominator = torch.sum(torch.abs(normalized_original) ** 2).clamp_min(1e-8)
    scale = item["scale"].cpu()

    # Undo sample-level scaling for display. The comparison remains native complex.
    original = normalized_original * scale
    reconstruction = reconstruction * scale
    fresh_reconstruction = fresh_reconstruction * scale
    error = reconstruction - original
    fresh_error = fresh_reconstruction - original
    denominator = torch.sum(torch.abs(original) ** 2).clamp_min(1e-8)

    return {
        "split": split,
        "index": index,
        "path": item["path"],
        "original": original[0].numpy(),
        "reconstruction": reconstruction[0].numpy(),
        "fresh_reconstruction": fresh_reconstruction[0].numpy(),
        "error": error[0].numpy(),
        "fresh_error": fresh_error[0].numpy(),
        "mse": float(torch.mean(torch.abs(error) ** 2).item()),
        "nmse": float((torch.sum(torch.abs(error) ** 2) / denominator).item()),
        "fresh_nmse": float((torch.sum(torch.abs(fresh_error) ** 2) / denominator).item()),
        "normalized_nmse": float((torch.sum(torch.abs(normalized_error) ** 2) / normalized_denominator).item()),
        "normalized_fresh_nmse": float((torch.sum(torch.abs(normalized_fresh_error) ** 2) / normalized_denominator).item()),
        "max_abs_reconstruction_minus_original": float(torch.max(torch.abs(error)).item()),
        "normalized_max_abs_reconstruction_minus_original": float(torch.max(torch.abs(normalized_error)).item()),
        "mean_abs_reconstruction_minus_fresh": float(torch.mean(torch.abs(reconstruction - fresh_reconstruction)).item()),
        "normalized_mean_abs_reconstruction_minus_fresh": float(torch.mean(torch.abs(reconstruction / scale - fresh_reconstruction / scale)).item()),
    }


selected = []
for idx in evenly_spaced_indices(len(test_dataset), TEST_SAMPLES):
    selected.append(reconstruct_item(test_dataset, "test", idx))
for idx in evenly_spaced_indices(len(train_dataset), TRAIN_SAMPLES):
    selected.append(reconstruct_item(train_dataset, "train", idx))

for row in selected:
    print(
        f"{row['split']:>5} index={row['index']:>5} "
        f"loaded_nmse={row['nmse']:.6f} fresh_nmse={row['fresh_nmse']:.6f} "
        f"norm_max_abs_loaded_minus_original={row['normalized_max_abs_reconstruction_minus_original']:.6f} "
        f"norm_mean_abs_loaded_minus_fresh={row['normalized_mean_abs_reconstruction_minus_fresh']:.6f} "
        f"file={Path(row['path']).name}"
    )

## Magnitude Comparison

Each row shows original magnitude, trained-checkpoint reconstruction, fresh random-weight reconstruction, trained absolute error, and fresh-model absolute error.

The absolute-error panel intentionally uses its own robust color scale so residual structure is visible. Use the printed NMSE values above to judge the actual size of the residual.

In [ ]:
def image_panel(array: np.ndarray, title: str, clim: tuple[float, float] | None = None, cmap: str = "gray"):
    image = hv.Image(np.asarray(array))
    opts = dict(width=300, height=300, cmap=cmap, colorbar=True, title=title, axiswise=True)
    if clim is not None:
        opts["clim"] = clim
    return image.opts(**opts)


def magnitude_row(sample: dict[str, object]):
    original = np.abs(sample["original"])
    reconstruction = np.abs(sample["reconstruction"])
    fresh_reconstruction = np.abs(sample["fresh_reconstruction"])
    error = np.abs(sample["error"])
    fresh_error = np.abs(sample["fresh_error"])
    shared_high = float(np.percentile(np.concatenate([original.ravel(), reconstruction.ravel(), fresh_reconstruction.ravel()]), 99.5))
    trained_error_high = float(np.percentile(error.ravel(), 99.5))
    fresh_error_high = float(np.percentile(fresh_error.ravel(), 99.5))
    label = f"{sample['split']} idx={sample['index']} trained_nmse={sample['nmse']:.4f} fresh_nmse={sample['fresh_nmse']:.4f}"
    return (
        image_panel(original, f"Original | {label}", clim=(0.0, shared_high))
        + image_panel(reconstruction, f"Trained Reconstruction | {Path(sample['path']).name}", clim=(0.0, shared_high))
        + image_panel(fresh_reconstruction, "Fresh Random Reconstruction", clim=(0.0, shared_high))
        + image_panel(error, "Trained Abs Error", clim=(0.0, trained_error_high), cmap="magma")
        + image_panel(fresh_error, "Fresh Abs Error", clim=(0.0, fresh_error_high), cmap="magma")
    )


magnitude_layout = hv.Layout([panel for sample in selected for panel in magnitude_row(sample)]).cols(5)
magnitude_layout

## Phase Comparison

The phase view is useful for checking whether the complex-valued model is preserving angular structure rather than only matching magnitude. The fresh random-weight phase is included as a visual baseline.

In [ ]:
def phase_row(sample: dict[str, object]):
    original_phase = np.angle(sample["original"])
    reconstruction_phase = np.angle(sample["reconstruction"])
    fresh_reconstruction_phase = np.angle(sample["fresh_reconstruction"])
    phase_error = np.angle(sample["reconstruction"] * np.conj(sample["original"]))
    fresh_phase_error = np.angle(sample["fresh_reconstruction"] * np.conj(sample["original"]))
    label = f"{sample['split']} idx={sample['index']}"
    return (
        image_panel(original_phase, f"Original Phase | {label}", clim=(-np.pi, np.pi), cmap="twilight")
        + image_panel(reconstruction_phase, f"Trained Phase | {Path(sample['path']).name}", clim=(-np.pi, np.pi), cmap="twilight")
        + image_panel(fresh_reconstruction_phase, "Fresh Random Phase", clim=(-np.pi, np.pi), cmap="twilight")
        + image_panel(phase_error, "Trained Wrapped Phase Error", clim=(-np.pi, np.pi), cmap="twilight")
        + image_panel(fresh_phase_error, "Fresh Wrapped Phase Error", clim=(-np.pi, np.pi), cmap="twilight")
    )


phase_layout = hv.Layout([panel for sample in selected for panel in phase_row(sample)]).cols(5)
phase_layout

## Complex-Plane Scatter

These plots sample pixels from each complex image and show imaginary vs real values. The first row compares the distribution of complex values for original, trained reconstruction, and fresh random reconstruction. The second row compares original values directly against reconstructed values; points near the diagonal indicate better agreement.

In [ ]:
SCATTER_POINTS = 5000


def sample_complex_values(array: np.ndarray, count: int = SCATTER_POINTS) -> np.ndarray:
    values = np.asarray(array).reshape(-1)
    finite = np.isfinite(values.real) & np.isfinite(values.imag)
    values = values[finite]
    if values.size <= count:
        return values
    rng = np.random.default_rng(123)
    return values[rng.choice(values.size, size=count, replace=False)]


def complex_distribution_scatter(values: np.ndarray, title: str, color: str, axis_limit: float):
    frame = pd.DataFrame({"real": values.real, "imag": values.imag})
    return hv.Scatter(frame, kdims="real", vdims="imag", label=title).opts(
        width=280,
        height=280,
        color=color,
        alpha=0.25,
        size=2,
        xlim=(-axis_limit, axis_limit),
        ylim=(-axis_limit, axis_limit),
        xlabel="real",
        ylabel="imag",
        title=title,
    ) * hv.HLine(0).opts(color="black", line_width=1, line_alpha=0.35) * hv.VLine(0).opts(color="black", line_width=1, line_alpha=0.35)


def paired_real_imag_scatter(original: np.ndarray, reconstruction: np.ndarray, title: str, color: str, axis_limit: float):
    original_values = np.asarray(original).reshape(-1)
    reconstruction_values = np.asarray(reconstruction).reshape(-1)
    finite = (
        np.isfinite(original_values.real)
        & np.isfinite(original_values.imag)
        & np.isfinite(reconstruction_values.real)
        & np.isfinite(reconstruction_values.imag)
    )
    original_values = original_values[finite]
    reconstruction_values = reconstruction_values[finite]
    if original_values.size > SCATTER_POINTS:
        rng = np.random.default_rng(123)
        idx = rng.choice(original_values.size, size=SCATTER_POINTS, replace=False)
        original_values = original_values[idx]
        reconstruction_values = reconstruction_values[idx]

    real_frame = pd.DataFrame({"original": original_values.real, "reconstruction": reconstruction_values.real})
    imag_frame = pd.DataFrame({"original": original_values.imag, "reconstruction": reconstruction_values.imag})
    diagonal = hv.Curve([(-axis_limit, -axis_limit), (axis_limit, axis_limit)], kdims="original", vdims="reconstruction")
    real_plot = hv.Scatter(real_frame, kdims="original", vdims="reconstruction", label=f"{title} real").opts(
        width=280,
        height=280,
        color=color,
        alpha=0.25,
        size=2,
        xlim=(-axis_limit, axis_limit),
        ylim=(-axis_limit, axis_limit),
        xlabel="original real",
        ylabel="reconstructed real",
        title=f"{title}: real agreement",
    ) * diagonal.opts(color="black", line_dash="dashed", line_width=1)
    imag_plot = hv.Scatter(imag_frame, kdims="original", vdims="reconstruction", label=f"{title} imag").opts(
        width=280,
        height=280,
        color=color,
        alpha=0.25,
        size=2,
        xlim=(-axis_limit, axis_limit),
        ylim=(-axis_limit, axis_limit),
        xlabel="original imag",
        ylabel="reconstructed imag",
        title=f"{title}: imag agreement",
    ) * diagonal.opts(color="black", line_dash="dashed", line_width=1)
    return real_plot + imag_plot


def complex_scatter_row(sample: dict[str, object]):
    original_values = sample_complex_values(sample["original"])
    trained_values = sample_complex_values(sample["reconstruction"])
    fresh_values = sample_complex_values(sample["fresh_reconstruction"])
    all_values = np.concatenate([original_values, trained_values, fresh_values])
    axis_limit = float(np.percentile(np.abs(np.concatenate([all_values.real, all_values.imag])), 99.5))
    axis_limit = max(axis_limit, 1e-8)
    label = f"{sample['split']} idx={sample['index']}"
    distribution = (
        complex_distribution_scatter(original_values, f"Original complex plane | {label}", "#0072B2", axis_limit)
        + complex_distribution_scatter(trained_values, "Trained complex plane", "#D55E00", axis_limit)
        + complex_distribution_scatter(fresh_values, "Fresh complex plane", "#009E73", axis_limit)
    )
    trained_agreement = paired_real_imag_scatter(sample["original"], sample["reconstruction"], "Trained", "#D55E00", axis_limit)
    fresh_agreement = paired_real_imag_scatter(sample["original"], sample["fresh_reconstruction"], "Fresh", "#009E73", axis_limit)
    return (distribution + trained_agreement + fresh_agreement).cols(3)


complex_scatter_layout = hv.Layout([panel for sample in selected for panel in complex_scatter_row(sample)]).cols(3)
complex_scatter_layout

## Optional HTML Export

In [ ]:
EXPORT_HTML = False
EXPORT_PATH = REPO_ROOT / "runs" / "complex_t2_autoencoder" / "reconstruction_selection.html"

if EXPORT_HTML:
    hv.save((magnitude_layout + phase_layout).cols(1), EXPORT_PATH)
    print(f"wrote {EXPORT_PATH}")